# Building Machine Learning Classifiers: Evaluate Gradient Boosting with GridSearchCV

**Grid-search:** Exhaustively search all parameter combinations in a given grid to determine the best model.

**Cross-validation:** Divide a dataset into k subsets and repeat the holdout method k times where a different subset is used as the holdout set in each iteration.

### Read in text

In [1]:
import nltk
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import string

stopwords = nltk.corpus.stopwords.words('english')
ps = nltk.PorterStemmer()

data = pd.read_csv("SMSSpamCollection.tsv", sep='\t')
data.columns = ['label', 'body_text']

def count_punct(text):
    count = sum([1 for char in text if char in string.punctuation])
    return round(count/(len(text) - text.count(" ")), 3)*100

data['body_len'] = data['body_text'].apply(lambda x: len(x) - x.count(" "))
data['punct%'] = data['body_text'].apply(lambda x: count_punct(x))

def clean_text(text):
    text = "".join([word.lower() for word in text if word not in string.punctuation])
    tokens = re.split(r'\W+', text)
    text = [ps.stem(word) for word in tokens if word not in stopwords]
    return text

# TF-IDF
tfidf_vect = TfidfVectorizer(analyzer=clean_text)
X_tfidf = tfidf_vect.fit_transform(data['body_text'])
X_tfidf_feat = pd.concat([data['body_len'], data['punct%'], pd.DataFrame(X_tfidf.toarray())], axis=1)
X_tfidf_feat.columns = X_tfidf_feat.columns.astype(str)

# CountVectorizer
count_vect = CountVectorizer(analyzer=clean_text)
X_count = count_vect.fit_transform(data['body_text'])
X_count_feat = pd.concat([data['body_len'], data['punct%'], pd.DataFrame(X_count.toarray())], axis=1)
X_count_feat.columns = X_count_feat.columns.astype(str)

X_count_feat.head()

,body_len,punct%,0,1,2,3,4,5,6,7,...,8094,8095,8096,8097,8098,8099,8100,8101,8102,8103
0,128,4.7,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,49,4.1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,62,3.2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,28,7.1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,135,4.4,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Exploring parameter settings using GridSearchCV

In [2]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

In [3]:
gb = GradientBoostingClassifier()
param = {
    'n_estimators': [100, 150],
    'max_depth': [7, 11, 15],
    'learning_rate': [0.1]
}

In [4]:
# gs = GridSearchCV(gb, param, cv=5, n_jobs=-1, verbose=2)
# cv_fit = gs.fit(X_tfidf_feat, data['label'])
# pd.DataFrame(cv_fit.cv_results_).sort_values('mean_test_score', ascending=False)[0:5]

In [5]:
# gs = GridSearchCV(gb, param, cv=5, n_jobs=-1, verbose=2)
# cv_fit = gs.fit(X_count_feat, data['label'])
# pd.DataFrame(cv_fit.cv_results_).sort_values('mean_test_score', ascending=False)[0:5]

In [6]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_enc = le.fit_transform(data['label'])

xgb = XGBClassifier(
    device='cuda',
    eval_metric='logloss',
    verbosity=0
)

param = {
    'n_estimators': [100, 150],
    'max_depth': [7, 11, 15],
    'learning_rate': [0.1]
}

gs = GridSearchCV(xgb, param, cv=5, n_jobs=-1, verbose=2)

In [7]:
cv_fit = gs.fit(X_tfidf_feat, y_enc)
pd.DataFrame(cv_fit.cv_results_).sort_values('mean_test_score', ascending=False)[0:5]

Fitting 5 folds for each of 6 candidates, totalling 30 fits
[CV] END ...learning_rate=0.1, max_depth=7, n_estimators=100; total time=   5.8s
[CV] END ..learning_rate=0.1, max_depth=15, n_estimators=100; total time=   7.1s
[CV] END ...learning_rate=0.1, max_depth=7, n_estimators=100; total time=   6.0s
[CV] END ..learning_rate=0.1, max_depth=15, n_estimators=100; total time=   7.0s
[CV] END ..learning_rate=0.1, max_depth=11, n_estimators=100; total time=   6.4s
[CV] END ..learning_rate=0.1, max_depth=15, n_estimators=100; total time=   6.8s
[CV] END ..learning_rate=0.1, max_depth=11, n_estimators=150; total time=   8.4s
[CV] END ..learning_rate=0.1, max_depth=15, n_estimators=100; total time=   6.5s
[CV] END ...learning_rate=0.1, max_depth=7, n_estimators=150; total time=   8.8s
[CV] END ..learning_rate=0.1, max_depth=15, n_estimators=150; total time=   9.9s
[CV] END ..learning_rate=0.1, max_depth=11, n_estimators=150; total time=  10.0s
[CV] END ..learning_rate=0.1, max_depth=15, n_est

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_learning_rate,param_max_depth,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
1,11.529487,2.518671,0.222012,0.057678,0.1,7,150,"{'learning_rate': 0.1, 'max_depth': 7, 'n_esti...",0.973968,0.982047,0.977538,0.964960,0.971249,0.973952,0.005772,1
2,11.325230,2.766616,0.308305,0.057744,0.1,11,100,"{'learning_rate': 0.1, 'max_depth': 11, 'n_est...",0.974865,0.977558,0.978437,0.964960,0.968553,0.972875,0.005257,2
3,10.304353,1.976395,0.214425,0.025782,0.1,11,150,"{'learning_rate': 0.1, 'max_depth': 11, 'n_est...",0.974865,0.978456,0.977538,0.967655,0.964960,0.972695,0.005415,3
5,9.646375,0.156893,0.236654,0.007391,0.1,15,150,"{'learning_rate': 0.1, 'max_depth': 15, 'n_est...",0.974865,0.976661,0.976640,0.964960,0.968553,0.972336,0.004740,4
4,6.615110,0.210417,0.234929,0.013558,0.1,15,100,"{'learning_rate': 0.1, 'max_depth': 15, 'n_est...",0.974865,0.978456,0.976640,0.963163,0.966757,0.971976,0.005950,5


In [8]:
cv_fit = gs.fit(X_count_feat, y_enc)
pd.DataFrame(cv_fit.cv_results_).sort_values('mean_test_score', ascending=False)[0:5]

Fitting 5 folds for each of 6 candidates, totalling 30 fits


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_learning_rate,param_max_depth,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
2,9.497728,2.010441,0.590341,0.130526,0.1,11,100,"{'learning_rate': 0.1, 'max_depth': 11, 'n_est...",0.976661,0.979354,0.977538,0.964960,0.972147,0.974132,0.005164,1
3,12.896050,2.264714,0.304077,0.010337,0.1,11,150,"{'learning_rate': 0.1, 'max_depth': 11, 'n_est...",0.973968,0.980251,0.977538,0.965858,0.972147,0.973953,0.004926,2
1,7.713586,1.664877,0.774065,0.261933,0.1,7,150,"{'learning_rate': 0.1, 'max_depth': 7, 'n_esti...",0.975763,0.977558,0.977538,0.965858,0.971249,0.973593,0.004502,3
5,7.903151,0.402638,0.296995,0.008755,0.1,15,150,"{'learning_rate': 0.1, 'max_depth': 15, 'n_est...",0.975763,0.977558,0.974843,0.966757,0.973046,0.973593,0.003716,4
4,5.761267,0.479885,0.305437,0.008850,0.1,15,100,"{'learning_rate': 0.1, 'max_depth': 15, 'n_est...",0.971275,0.975763,0.976640,0.964960,0.970350,0.971797,0.004202,5
